# OpenMed Quickstart: Clinical Text De-Identification

Welcome to the **OpenMed De-Identification Quickstart**. This interactive notebook walks through the primary de-identification techniques supported by OpenMed:

1. **Masking (`mask`)**: Replacing direct identifiers with category tags.
2. **Reversible Replacement (`replace`)**: Generating realistic synthetic stand-ins with recoverable lookup mappings.
3. **Cryptographic Hashing (`hash`)**: Computing keyed HMAC/SHA digests for safe cross-dataset entity linking.

> **Privacy Invariant**: All data in this notebook is completely **synthetic**. Never commit real Protected Health Information (PHI) to git repositories.

> **Important Note on Offline Demo Mode**:
> To guarantee that this notebook executes 100% offline in CI and local test suites without downloading multi-gigabyte Hugging Face model weights, the runtime below initializes `_NoDownloadLoader`.
> **In this offline demonstration, names are matched via a fixed offline demo list, not a live NER model.**
> In production environments with model weights installed (e.g. `uv pip install ".[hf]"`), OpenMed loads full transformer models (such as GLiNER or token-classification backends) to detect names and clinical entities dynamically across arbitrary text.

In [1]:
import json
import logging
from typing import Any
from openmed import deidentify, reidentify

# Suppress first-run download telemetry for offline execution
logging.getLogger("openmed.core.models").setLevel(logging.ERROR)

# Offline mock loader for token classification on synthetic demonstration notes.
# NOTE: Names are matched via a fixed offline demo list, not a live transformer NER model.
class _NoDownloadTokenClassificationPipeline:
    tokenizer = None
    def __call__(self, inputs: Any, **_: Any) -> list[Any]:
        def _extract(text: str) -> list[dict[str, Any]]:
            spans: list[dict[str, Any]] = []
            synthetic_names = ["John Doe", "Jane Roe", "Alice Smith", "Bob Jones", "David Miller", "Emma Watson"]
            for name in synthetic_names:
                idx = 0
                while True:
                    idx = text.find(name, idx)
                    if idx == -1:
                        break
                    spans.append({
                        "entity_group": "PERSON",
                        "start": idx,
                        "end": idx + len(name),
                        "score": 0.99,
                        "word": name,
                    })
                    idx += len(name)
            spans.sort(key=lambda s: s["start"])
            return spans
        if isinstance(inputs, list):
            return [_extract(t) for t in inputs]
        return _extract(inputs) if isinstance(inputs, str) else []

class _NoDownloadLoader:
    config = None
    def create_pipeline(self, *_: Any, **__: Any) -> Any:
        return _NoDownloadTokenClassificationPipeline()
    def get_max_sequence_length(self, *_: Any, **__: Any) -> None:
        return None

OFFLINE_LOADER = _NoDownloadLoader()
print("OpenMed de-identification runtime initialized successfully (offline mode).")

OpenMed de-identification runtime initialized successfully (offline mode).


## 1. Define Synthetic Clinical Input

We define a sample clinical note containing direct patient identifiers (Name, DOB, Phone, Email, MRN).

In [2]:
SYNTHETIC_NOTE = (
    "Patient John Doe (DOB 01/15/1970) called from 555-123-4567. "
    "Contact Jane Roe at jane.roe@example.test regarding MRN 00123456."
)

print("=== Synthetic Input Note ===")
print(SYNTHETIC_NOTE)

=== Synthetic Input Note ===
Patient John Doe (DOB 01/15/1970) called from 555-123-4567. Contact Jane Roe at jane.roe@example.test regarding MRN 00123456.


## 2. Redaction via Masking (`method='mask'`)

Masking replaces detected PHI spans with standardized category tags like `[PERSON]`, `[DATE]`, `[PHONE]`, and `[EMAIL]`.

In [3]:
masked_result = deidentify(
    SYNTHETIC_NOTE,
    method="mask",
    loader=OFFLINE_LOADER,
    use_safety_sweep=True,
)

print("=== Masked Output ===")
print(masked_result.deidentified_text)

=== Masked Output ===
Patient [PERSON] (DOB [date]) called from [phone_number]. Contact [PERSON] at [email] regarding [medical_record_number].


## 3. Reversible Synthetic Replacement (`method='replace'`)

Replacement substitutes detected identifiers with realistic synthetic stand-ins generated via `Faker`. When `keep_mapping=True`, OpenMed captures the surrogate mapping so authorized users can reverse the redaction with `reidentify()`.

In [4]:
replaced_result = deidentify(
    SYNTHETIC_NOTE,
    method="replace",
    seed=42,
    keep_mapping=True,
    loader=OFFLINE_LOADER,
    use_safety_sweep=True,
)

print("=== Replaced Output ===")
print(replaced_result.deidentified_text)

print("\n=== Replacement Mapping ===")
print(json.dumps(replaced_result.mapping, indent=2, sort_keys=True))

# Verify round-trip restoration
restored_text = reidentify(replaced_result.deidentified_text, replaced_result.mapping)
print(f"\nRestored text matches original: {restored_text == SYNTHETIC_NOTE}")

=== Replaced Output ===
Patient Heather Reed (DOB 10/09/1999) called from 772-056-5883. Contact Robin Thomas at jamesharris@example.test regarding 147-46-7245.

=== Replacement Mapping ===
{
  "10/09/1999": "01/15/1970",
  "147-46-7245": "MRN 00123456",
  "772-056-5883": "555-123-4567",
  "Heather Reed": "John Doe",
  "Robin Thomas": "Jane Roe",
  "jamesharris@example.test": "jane.roe@example.test"
}

Restored text matches original: True


## 4. Cryptographic Hashing (`method='hash'`)

Hashing computes keyed deterministic digests (`date_<hash>`, `mrn_<hash>`) that preserve entity linkage across tables while permanently concealing raw values.

In [5]:
hashed_result = deidentify(
    SYNTHETIC_NOTE,
    method="hash",
    loader=OFFLINE_LOADER,
    use_safety_sweep=True,
)

print("=== Hashed Output ===")
print(hashed_result.deidentified_text)

=== Hashed Output ===
Patient PERSON_6cea57c2 (DOB date_81ce177e) called from phone_number_d36e8308. Contact PERSON_9231c165 at email_d5de0402 regarding medical_record_number_4f0528a2.
